In [ ]:
# ==============================================================================# 🚀 DO NOT MODIFY: Standardized Notebook Setup# ==============================================================================# This cell is designed to work in both Google Colab and local environments.# It ensures that the environment is correctly configured by cloning (or# locating) the project repository and installing the necessary dependencies.## ------------------------------------------------------------------------------##  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):##  This cell will automatically find the repository root and configure your#  environment. Just make sure you have run: pip install -e .[dev]## ------------------------------------------------------------------------------import osimport subprocessimport sysfrom pathlib import Path# --- Configuration ---REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned# --- End of Configuration ---def find_repo_root(start_path: Path) -> Path | None:    """    Find the repository root by looking for pyproject.toml.    Searches upward from start_path until it finds pyproject.toml or hits root.    Args:        start_path: Directory to start searching from.    Returns:        Path to repository root, or None if not found.    """    current = start_path.resolve()    while current \!= current.parent:  # Stop at filesystem root        if (current / "pyproject.toml").exists():            return current        current = current.parent    return Nonedef detect_active_branch(repo_dir: Path) -> str:    """    Determine the active git branch for pulling updates.    Tries multiple methods to detect the current branch name.    Args:        repo_dir: Path to the git repository.    Returns:        Branch name (defaults to 'master' if detection fails).    """    commands = [        "git symbolic-ref --short HEAD",        "git rev-parse --abbrev-ref HEAD",    ]    for cmd in commands:        result = subprocess.run(            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True        )        if result.returncode == 0:            branch = result.stdout.strip()            if branch and not branch.startswith("origin/"):                return branch    return "master"def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:    """    Run a shell command and raise an error if it fails.    Args:        cmd: The command to run.        cwd: Optional working directory for the command.    Raises:        RuntimeError: If the command returns a non-zero exit code.    """    result = subprocess.run(cmd, shell=True, cwd=cwd)    if result.returncode \!= 0:        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")# --- Detect environment ---try:    import google.colab  # noqa: F401    IN_COLAB = Trueexcept ImportError:    IN_COLAB = False# --- Main setup logic ---if IN_COLAB:    print("☁️  Running in Google Colab. Setting up the environment...")    # Determine repository path    start_dir = Path.cwd()    if start_dir.name == REPO_DIR.name:        repo_path = start_dir    else:        repo_path = start_dir / REPO_DIR    # Clone or update repository    if not repo_path.exists():        print(f"📥 Cloning repository from {REPO_URL}...")        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")        print(f"✅ Repository cloned to {repo_path}")    else:        print(f"📂 Repository already exists at {repo_path}")        active_branch = detect_active_branch(repo_path)        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)        print(f"✅ Repository updated")    # Verify repository structure    if not (repo_path / "pyproject.toml").exists():        raise FileNotFoundError(            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "            "The repository may be corrupted."        )    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    # Install dependencies (smart installation - only installs missing packages)    from core.notebook.setup import smart_install_dependencies    result = smart_install_dependencies(        repo_path=repo_path,        include_dev=True,        verbose=True,    )    # Fail loudly if critical packages failed to install    if result["failed"]:        print(f"⚠️  WARNING: {len(result['failed'])} packages failed to install:")        for pkg in result["failed"]:            print(f"  - {pkg}")        print("You may encounter import errors. Please check your internet connection.")    print("" + "=" * 70)    print("✅ Environment setup complete\! You can now proceed with the notebook.")    print("=" * 70)else:    print("💻 Running in local environment. Configuring...")    # Find the repository root    repo_path = find_repo_root(Path.cwd())    if repo_path is None:        raise FileNotFoundError(            "Could not find repository root (no pyproject.toml found). "            "Please ensure you are running this notebook from within the "            "ADH-LLM-Tutorials-2025 repository directory."        )    print(f"✅ Found repository root: {repo_path}")    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    print("" + "=" * 70)    print("✅ Local environment configured successfully\!")    print("=" * 70)    print("⚠️  Please ensure you have run: pip install -e .[dev]")    print("   (Required for local development)")

# 06b - Systematic Model Improvement: Hyperparameter Tuning

## Automating the Search for Better Models

In previous notebooks, we manually configured hyperparameters for our sequence models. While this approach provides valuable intuition, it's inefficient and unlikely to find optimal configurations.

In this notebook, we'll use **Optuna**, a powerful hyperparameter optimization framework, to systematically search for better model configurations. Because we invested in a robust, reusable `Trainer` class, we can wrap it in an optimization objective with minimal boilerplate code.

### What You'll Learn

1. How to define an optimization objective function that Optuna can optimize
2. How to use `trial.suggest_*` methods to sample hyperparameters
3. How to analyze optimization results with visualization
4. The power of the DRY principle: reusing our entire training engine for a new advanced task

In [ ]:
# Import required libraries
from pathlib import Path

import optuna
import torch
from torch.utils.data import DataLoader

from core.config import GRUConfig, TrainConfig
from core.data.loaders import SepsisDataset, sepsis_collate_fn, split_sepsis_dataset
from core.data.physionet_sepsis import get_sepsis_data
from core.models import GRUModel
from core.notebook import ensure_project_root
from core.train import Trainer

# Suppress optuna's verbose logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Step 1: Load Data

We'll load the PhysioNet dataset once and reuse it across all optimization trials.

In [ ]:
project_root = ensure_project_root()

# Load the preprocessed PhysioNet dataset
sepsis_df = get_sepsis_data()

num_patients = sepsis_df["patient_id"].nunique()
print(f"Total ICU patient stays: {num_patients:,}")
print(f"Total hourly measurements: {len(sepsis_df):,}")

### Cache the dataset once

To keep Optuna trials efficient, we materialize the `SepsisDataset` and 
deterministic train/validation split a single time. Each trial will reuse 
these subsets instead of rebuilding tensors from scratch.


In [ ]:
# Build dataset and deterministic split once
sepsis_dataset = SepsisDataset(sepsis_df)
TRAIN_VAL_SPLIT = 0.8
SPLIT_SEED = 42
train_subset, val_subset = split_sepsis_dataset(
    sepsis_dataset,
    train_val_split=TRAIN_VAL_SPLIT,
    seed=SPLIT_SEED,
)


def build_dataloaders(batch_size: int) -> tuple[DataLoader, DataLoader]:
    return (
        DataLoader(
            train_subset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=sepsis_collate_fn,
        ),
        DataLoader(
            val_subset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=sepsis_collate_fn,
        ),
    )

## Step 2: Define the Objective Function

The objective function is what Optuna will try to optimize. For each trial, Optuna will:

1. Suggest a set of hyperparameters
2. Call our objective function with those hyperparameters
3. Receive a score (the validation AUPRC)
4. Use that score to intelligently suggest the next set of hyperparameters

This is much smarter than random search or grid search because Optuna learns from previous trials.

💡 Instead of returning only the final epoch's metric, we track the best validation AUPRC observed during training. This avoids penalizing trials that briefly achieve excellent performance before overfitting.

⚠️  The Optuna study depends on tracking validation AUPRC during training, so the `TrainConfig` explicitly requests `val_metrics=("auprc",)`.


In [ ]:
def objective(trial: optuna.Trial) -> float:
    """
    Objective function for Optuna hyperparameter optimization.

    This function is called once per trial. Optuna will suggest hyperparameters,
    we train a model with those hyperparameters, and return the validation AUPRC.

    Parameters
    ----------
    trial : optuna.Trial
        The trial object that suggests hyperparameters.

    Returns
    -------
    float
        The validation AUPRC score for this trial.
    """
    # Sample hyperparameters from defined search spaces
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    max_grad_norm = trial.suggest_float("max_grad_norm", 0.5, 5.0)

    # For speed, we'll use fewer epochs during tuning
    epochs = 10

    # Create model configuration with sampled hyperparameters
    model_config = GRUConfig(
        input_size=34,  # Fixed by dataset
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    )

    # Create training configuration with sampled hyperparameters
    train_config = TrainConfig(
        learning_rate=learning_rate,
        epochs=epochs,
        batch_size=batch_size,
        device="cuda" if torch.cuda.is_available() else "cpu",
        optimizer="adam",
        loss="bce_logits",
        scheduler="none",
        max_grad_norm=max_grad_norm,
        train_val_split=TRAIN_VAL_SPLIT,
        split_seed=SPLIT_SEED,  # Keep split consistent across trials
        val_metrics=("auprc",),  # Ensure AUPRC is tracked each epoch
    )

    # Create data loaders with this trial's batch size
    train_loader, val_loader = build_dataloaders(batch_size=batch_size)

    # Create model and trainer
    model = GRUModel(model_config)
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=train_config,
        save_path=None,  # Don't save checkpoints during tuning
    )

    # Train the model
    history, _ = trainer.fit()

    val_metrics = history.get("val_metrics", {})
    auprc_history = val_metrics.get("auprc")
    if not auprc_history:
        raise RuntimeError("Trainer did not record AUPRC for this trial.")
    best_auprc = max(auprc_history)

    # Return the best validation score observed during training
    return best_auprc

## Step 3: Run the Optimization Study

Now we create an Optuna study and run the optimization. Each trial trains a complete model with different hyperparameters.

**Note:** This will take some time to run (approximately 10-20 minutes for 20 trials, depending on your hardware). For a quick demonstration, you can reduce `n_trials` to 5.

In [ ]:
# Create a study that maximizes the validation AUPRC
study = optuna.create_study(
    direction="maximize",
    study_name="gru_hyperparameter_tuning",
)

# Run the optimization
print("🔍 Starting hyperparameter optimization...")
print("This may take 10-20 minutes depending on your hardware.\n")

study.optimize(objective, n_trials=5, show_progress_bar=True)

print("\n✅ Optimization complete!")

## Step 4: Analyze the Results

Let's examine what Optuna discovered about our hyperparameter space.

In [ ]:
# Display the best trial
best_trial = study.best_trial

print("🏆 Best Trial Results")
print("=" * 60)
print(f"  Validation AUPRC: {best_trial.value:.4f}")
print(f"  Trial Number: {best_trial.number}")
print("\n📊 Best Hyperparameters:")
print("-" * 60)
for key, value in best_trial.params.items():
    print(f"  {key:20s}: {value}")

## Step 5: Visualize the Optimization Process

Optuna provides powerful visualization tools to understand how the optimization progressed.

In [ ]:
# Plot optimization history
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(
    title="Optimization History: AUPRC Over Trials",
    xaxis_title="Trial Number",
    yaxis_title="Validation AUPRC",
)
fig.show()

print(
    """
This plot shows how the best AUPRC improved over time. The blue line shows
individual trial results, while the red line shows the best score found so far.
"""
)

In [ ]:
# Plot parameter importances
fig = optuna.visualization.plot_param_importances(study)
fig.update_layout(
    title="Hyperparameter Importance",
    xaxis_title="Importance",
)
fig.show()

print(
    """
This plot shows which hyperparameters had the biggest impact on performance.
Hyperparameters with higher importance scores should be tuned more carefully.
"""
)

In [ ]:
# Plot parallel coordinate plot
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.update_layout(
    title="Parallel Coordinate Plot: Hyperparameter Relationships",
)
fig.show()

print(
    """
This plot shows the relationships between hyperparameters and performance.
Each line represents one trial. Lines are colored by their AUPRC score
(warmer colors = better performance).
"""
)

## Interpretation and Next Steps

### Key Insights from Hyperparameter Tuning

1. **Optimization Efficiency**: Optuna intelligently explores the hyperparameter space, often finding good configurations within the first few trials and then refining them.

2. **Parameter Importance**: The importance plot reveals which hyperparameters matter most. In many cases, learning rate and model capacity (hidden_size, num_layers) are the most influential.

3. **Interaction Effects**: The parallel coordinate plot can reveal interactions between parameters. For example, larger models may require more aggressive regularization (higher dropout).

### Practical Recommendations

- **Save the best model**: In practice, you would train a final model with the best hyperparameters and save it for deployment.
- **Cross-validation**: For more robust results, consider using k-fold cross-validation instead of a single train/val split.
- **Search space refinement**: After an initial broad search, you can narrow the search space around promising regions.
- **Multi-objective optimization**: Optuna also supports optimizing multiple objectives (e.g., maximizing AUPRC while minimizing model size).

### The Power of Abstraction

Notice how little code we needed to add hyperparameter tuning to our project. This is the payoff of investing in clean abstractions (the `Trainer` class, Pydantic configs, etc.). We reused our entire training pipeline for a sophisticated optimization task without duplicating any training logic.

## 🎯 Challenge Exercise

Try modifying the objective function to:

1. Tune a different model (LSTM or Transformer)
2. Add additional hyperparameters to the search space (e.g., optimizer choice, learning rate scheduling)
3. Change the optimization metric (e.g., use AUROC instead of AUPRC)
4. Implement early stopping using Optuna's pruning feature (`trial.report()` and `trial.should_prune()`)

Can you find hyperparameters that achieve better performance than our manual configurations?